# STN_0003 Demand Modeling Notebook (Restructured)

This notebook is a cleaned and parameterised rewrite of the original `09_stn0004_model.ipynb`, redesigned specifically for **`STN_0003`**.

## What was restructured from the original notebook

The original file mixed setup, station lookup, exploration, modeling, and presentation cells in a way that made it hard to reuse. Main issues found:

1. **Station-specific hardcoding everywhere**
   - `STN_0004` was embedded in multiple cells, plot titles, and comments.
   - Station identification used both direct ID filtering and coordinate matching, which is redundant.

2. **Repeated imports and Spark setup**
   - Imports and Spark initialization appeared in multiple places.
   - This increases notebook fragility and makes reruns harder.

3. **Exploration mixed with production modeling**
   - File-search / lookup cells and model training were interleaved.
   - Presentation commentary was mixed into computational cells.

4. **Weak portability**
   - Local absolute paths were hardcoded.
   - Some assumptions about column names were not validated before training.

5. **Model pipeline not modular**
   - Feature engineering logic was split across separate cells and versions.
   - There was no single reusable function for feature generation, train/test split, evaluation, and plotting.

6. **Metric reporting was partially hardcoded**
   - Some printed comparisons referenced fixed numbers instead of computed baseline values.

## New structure in this notebook

1. Configuration
2. Spark session setup
3. Data loading with schema validation
4. Station filtering for `STN_0003`
5. Quick data-quality checks
6. Reusable feature engineering
7. Train/test split
8. Baseline linear regression
9. Gradient boosted tree model
10. Model comparison
11. Diagnostics and plots
12. Recommendations / next steps

## Expected outcome

This version is easier to:
- retarget to another station,
- debug feature issues,
- compare baseline vs stronger models,
- keep as a reusable modeling template.

In [2]:

# =========================
# 1) Configuration
# =========================
from pathlib import Path

STATION_ID = "STN_0003"
TARGET_COL = "station_outflow"

# Update these paths for your machine if needed
FEATURES_PATH = "../../data/gold/bixi_model_features_v1"
MAPPING_FILE_PATH = "../../data/silver/station_cleaning/station_canonical_summary/part-00000-38378e28-0e5c-4f09-aa18-bda21ebb2469-c000.snappy.parquet"

TRAIN_YEAR = 2024
TEST_YEAR = 2025

print("Station:", STATION_ID)
print("Features path:", FEATURES_PATH)
print("Mapping path:", MAPPING_FILE_PATH)

Station: STN_0003
Features path: ../../data/gold/bixi_model_features_v1
Mapping path: ../../data/silver/station_cleaning/station_canonical_summary/part-00000-38378e28-0e5c-4f09-aa18-bda21ebb2469-c000.snappy.parquet


In [3]:

# =========================
# 2) Imports and Spark session
# =========================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.window import Window

from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression, GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator

JAVA_FLAGS = (
    "--add-opens=java.base/java.lang=ALL-UNNAMED "
    "--add-opens=java.base/java.lang.invoke=ALL-UNNAMED "
    "--add-opens=java.base/java.lang.reflect=ALL-UNNAMED "
    "--add-opens=java.base/java.io=ALL-UNNAMED "
    "--add-opens=java.base/java.net=ALL-UNNAMED "
    "--add-opens=java.base/java.nio=ALL-UNNAMED "
    "--add-opens=java.base/java.util=ALL-UNNAMED "
    "--add-opens=java.base/java.util.concurrent=ALL-UNNAMED "
    "--add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED "
    "--add-opens=java.base/sun.nio.ch=ALL-UNNAMED "
    "--add-opens=java.base/sun.nio.cs=ALL-UNNAMED "
    "--add-opens=java.base/sun.security.action=ALL-UNNAMED "
    "--add-opens=java.base/sun.util.calendar=ALL-UNNAMED"
)

os.environ["PYSPARK_SUBMIT_ARGS"] = (
    f'--driver-java-options="{JAVA_FLAGS}" pyspark-shell'
)

spark = (
    SparkSession.builder
    .appName(f"bixi-{STATION_ID.lower()}-forecast")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.session.timeZone", "America/Toronto")
    .getOrCreate()
)

print("Spark version:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/04 00:51:24 WARN Utils: Your hostname, users-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.4.39 instead (on interface en0)
26/04/04 00:51:24 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/04 00:51:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.1.1


In [4]:

# =========================
# 3) Load source data with validation
# =========================
raw_sdf = (
    spark.read.parquet(f"file://{FEATURES_PATH}")
    .withColumnRenamed("station_id", "canonical_station_id")
    .withColumnRenamed("demand_count", TARGET_COL)
    .withColumn("ts_hour", F.to_timestamp("ts_hour"))
    .withColumn("ride_year", F.year("ts_hour"))
)

required_columns = {
    "canonical_station_id", "ts_hour", TARGET_COL,
    "temp", "precip", "is_holiday", "is_weekend", "day_of_week"
}
missing = sorted(required_columns - set(raw_sdf.columns))
if missing:
    raise ValueError(f"Missing required columns: {missing}")

print("Raw row count:", raw_sdf.count())
print("Schema validated.")
raw_sdf.select("canonical_station_id", "ts_hour", TARGET_COL).show(5, truncate=False)

26/04/04 00:52:06 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: file://../../data/gold/bixi_model_features_v1.
java.lang.IllegalArgumentException: Wrong FS: file://../../data/gold/bixi_model_features_v1, expected: file:///
	at org.apache.hadoop.fs.FileSystem.checkPath(FileSystem.java:824)
	at org.apache.hadoop.fs.RawLocalFileSystem.pathToFile(RawLocalFileSystem.java:125)
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:975)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384

IllegalArgumentException: Wrong FS: file://../../data/gold/bixi_model_features_v1, expected: file:///

26/04/04 07:54:14 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 710698 ms exceeds timeout 120000 ms
26/04/04 07:54:15 WARN SparkContext: Killing executors is not supported by current scheduler.
26/04/04 07:54:15 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$

In [ ]:

# =========================
# 4) Optional mapping lookup for metadata
# =========================
mapping_df = spark.read.parquet(f"file://{MAPPING_FILE_PATH}")

station_meta = (
    mapping_df
    .filter(F.trim(F.col("canonical_station_id")) == STATION_ID)
)

print(f"Metadata rows for {STATION_ID}: {station_meta.count()}")
station_meta.show(truncate=False)

In [ ]:

# =========================
# 5) Filter station-level series
# =========================
stn_data = (
    raw_sdf
    .filter(F.trim(F.col("canonical_station_id")) == STATION_ID)
    .orderBy("ts_hour")
)

row_count = stn_data.count()
print(f"Rows for {STATION_ID}: {row_count}")

if row_count == 0:
    raise ValueError(f"No records found for {STATION_ID}. Check the station ID or upstream mapping.")

stn_data.select("ts_hour", TARGET_COL, "temp", "precip").show(10, truncate=False)

In [ ]:

# =========================
# 6) Quick data-quality checks
# =========================
summary_df = stn_data.select(
    F.min("ts_hour").alias("min_ts"),
    F.max("ts_hour").alias("max_ts"),
    F.count("*").alias("n_rows"),
    F.sum(F.col(TARGET_COL).isNull().cast("int")).alias("target_nulls"),
    F.sum(F.col("temp").isNull().cast("int")).alias("temp_nulls"),
    F.sum(F.col("precip").isNull().cast("int")).alias("precip_nulls")
)

summary_df.show(truncate=False)

year_counts = stn_data.groupBy("ride_year").count().orderBy("ride_year")
year_counts.show()

## Detailed modeling plan

### A. Data preparation
- keep one clean station-level table for `STN_0003`,
- validate required columns before modeling,
- inspect yearly coverage to confirm both train and test years exist.

### B. Feature engineering
Use one reusable function to create:
- cyclical hour features,
- short-term lag (`1h`),
- daily lag (`24h`),
- weekly lag (`168h`),
- short rolling momentum (`3h average`).

This is cleaner than splitting feature engineering into separate notebook branches.

### C. Train/test design
- train on **2024**
- test on **2025**

This matches the intent of the original notebook while making the logic explicit.

### D. Model strategy
1. **Baseline:** Linear Regression  
   Useful to quantify how much gain comes from the nonlinear model.
2. **Primary model:** GBT Regressor  
   Better suited for nonlinear demand spikes and weather/time interactions.

### E. Evaluation
Report:
- MAE
- RMSE
- R²

Then compare models in one table rather than relying on hardcoded print statements.

### F. Diagnostics
Create:
- time series plot,
- actual vs predicted scatter,
- residual-by-hour summary,
- optional feature importance for GBT.

### G. Next improvement path
If STN_0003 still underperforms:
- add seasonal features,
- add nearby-station spillover features,
- try cross-validation or hyperparameter tuning,
- test inflow/outflow joint modeling.

In [ ]:

# =========================
# 7) Reusable feature engineering
# =========================
def build_features(station_sdf, target_col):
    win_order = Window.partitionBy("canonical_station_id").orderBy("ts_hour")
    win_3h = win_order.rowsBetween(-3, -1)

    featured = (
        station_sdf
        .withColumn("hour", F.hour("ts_hour"))
        .withColumn("hour_sin", F.sin(F.lit(2 * np.pi) * F.col("hour") / F.lit(24)))
        .withColumn("hour_cos", F.cos(F.lit(2 * np.pi) * F.col("hour") / F.lit(24)))
        .withColumn("lag_1h", F.lag(target_col, 1).over(win_order))
        .withColumn("lag_24h", F.lag(target_col, 24).over(win_order))
        .withColumn("lag_168h", F.lag(target_col, 168).over(win_order))
        .withColumn("rolling_3h_avg", F.avg(target_col).over(win_3h))
        .fillna({"temp": 15.0, "precip": 0.0})
        .dropna(subset=["lag_1h", "lag_24h", "lag_168h", "rolling_3h_avg"])
    )
    return featured

model_sdf = build_features(stn_data, TARGET_COL)
print("Feature-ready rows:", model_sdf.count())
print("Columns:", model_sdf.columns)

In [ ]:

# =========================
# 8) Train / test split
# =========================
feature_cols = [
    "temp", "precip", "is_holiday", "is_weekend", "day_of_week",
    "hour_sin", "hour_cos", "lag_1h", "lag_24h", "lag_168h", "rolling_3h_avg"
]

train_df = model_sdf.filter(F.col("ride_year") == TRAIN_YEAR)
test_df = model_sdf.filter(F.col("ride_year") == TEST_YEAR)

train_n = train_df.count()
test_n = test_df.count()

print(f"Train rows ({TRAIN_YEAR}):", train_n)
print(f"Test rows ({TEST_YEAR}):", test_n)

if train_n == 0 or test_n == 0:
    raise ValueError(
        f"Train/Test split failed. Found train={train_n}, test={test_n}. "
        "Check year coverage in the station-level data."
    )

In [ ]:

# =========================
# 9) Evaluation helper
# =========================
def evaluate_predictions(pred_df, label_col=TARGET_COL, prediction_col="prediction"):
    metrics = {}
    for metric_name in ["mae", "rmse", "r2"]:
        evaluator = RegressionEvaluator(
            labelCol=label_col,
            predictionCol=prediction_col,
            metricName=metric_name
        )
        metrics[metric_name.upper()] = evaluator.evaluate(pred_df)
    return metrics

In [ ]:

# =========================
# 10) Baseline model: Linear Regression
# =========================
lr_assembler = VectorAssembler(inputCols=feature_cols, outputCol="unscaled_features")
lr_scaler = StandardScaler(inputCol="unscaled_features", outputCol="features", withStd=True, withMean=True)
lr = LinearRegression(featuresCol="features", labelCol=TARGET_COL, predictionCol="prediction")

lr_pipeline = Pipeline(stages=[lr_assembler, lr_scaler, lr])
lr_model = lr_pipeline.fit(train_df)
lr_predictions = lr_model.transform(test_df)

lr_metrics = evaluate_predictions(lr_predictions)
lr_metrics

In [ ]:

# =========================
# 11) Main model: Gradient Boosted Trees
# =========================
gbt_assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
gbt = GBTRegressor(
    featuresCol="features",
    labelCol=TARGET_COL,
    predictionCol="prediction",
    maxIter=150,
    maxDepth=7,
    stepSize=0.05,
    seed=42
)

gbt_pipeline = Pipeline(stages=[gbt_assembler, gbt])
gbt_model = gbt_pipeline.fit(train_df)
gbt_predictions = gbt_model.transform(test_df)

gbt_metrics = evaluate_predictions(gbt_predictions)
gbt_metrics

In [ ]:

# =========================
# 12) Compare model performance
# =========================
comparison_pd = pd.DataFrame([
    {"model": "Linear Regression", **lr_metrics},
    {"model": "GBT Regressor", **gbt_metrics},
]).sort_values(by="MAE", ascending=True)

comparison_pd

In [ ]:

# =========================
# 13) Best model selection
# =========================
best_model_name = comparison_pd.iloc[0]["model"]
best_predictions = gbt_predictions if best_model_name == "GBT Regressor" else lr_predictions

print("Best model:", best_model_name)
comparison_pd

In [ ]:

# =========================
# 14) Time series plot
# =========================
plot_pd = (
    best_predictions
    .orderBy("ts_hour")
    .select("ts_hour", TARGET_COL, "prediction")
    .limit(24 * 14)
    .toPandas()
)

plt.figure(figsize=(15, 6))
plt.plot(plot_pd["ts_hour"], plot_pd[TARGET_COL], label=f"Actual {TEST_YEAR}")
plt.plot(plot_pd["ts_hour"], plot_pd["prediction"], label=f"Predicted {TEST_YEAR}", linestyle="--")
plt.title(f"{STATION_ID}: Hourly Outflow Prediction vs Actual")
plt.xlabel("Timestamp")
plt.ylabel("Bikes Outflowing")
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:

# =========================
# 15) Actual vs predicted scatter
# =========================
scatter_pd = (
    best_predictions
    .select(TARGET_COL, "prediction")
    .sample(False, 0.2, seed=42)
    .toPandas()
)

plt.figure(figsize=(8, 8))
plt.scatter(scatter_pd[TARGET_COL], scatter_pd["prediction"], alpha=0.4)
max_val = max(scatter_pd[TARGET_COL].max(), scatter_pd["prediction"].max())
plt.plot([0, max_val], [0, max_val], linestyle="--", label="Perfect prediction")
plt.title(f"{STATION_ID}: Actual vs Predicted")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:

# =========================
# 16) Residual diagnostics by day/hour
# =========================
residual_pd = (
    best_predictions
    .withColumn("residual", F.col(TARGET_COL) - F.col("prediction"))
    .withColumn("day_name", F.date_format(F.col("ts_hour"), "EEEE"))
    .withColumn("hour_int", F.hour(F.col("ts_hour")))
    .groupBy("day_name", "hour_int")
    .agg(F.avg("residual").alias("avg_residual"))
    .toPandas()
)

day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
residual_pd["day_name"] = pd.Categorical(residual_pd["day_name"], categories=day_order, ordered=True)
residual_pivot = residual_pd.pivot(index="day_name", columns="hour_int", values="avg_residual")

plt.figure(figsize=(16, 8))
plt.imshow(residual_pivot, aspect="auto")
plt.colorbar(label="Average residual (actual - prediction)")
plt.title(f"{STATION_ID}: Residual Pattern by Day and Hour")
plt.xlabel("Hour of day")
plt.ylabel("Day of week")
plt.xticks(range(len(residual_pivot.columns)), residual_pivot.columns)
plt.yticks(range(len(residual_pivot.index)), residual_pivot.index)
plt.tight_layout()
plt.show()

In [ ]:

# =========================
# 17) Feature importance (GBT only)
# =========================
if best_model_name == "GBT Regressor":
    gbt_stage = gbt_model.stages[-1]
    importances = list(gbt_stage.featureImportances)
    feature_importance_pd = (
        pd.DataFrame({"feature": feature_cols, "importance": importances})
        .sort_values("importance", ascending=False)
    )
    display(feature_importance_pd)

    plt.figure(figsize=(10, 5))
    plt.bar(feature_importance_pd["feature"], feature_importance_pd["importance"])
    plt.title(f"{STATION_ID}: GBT Feature Importances")
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Importance")
    plt.tight_layout()
    plt.show()
else:
    print("Feature importance chart skipped because the best model was not GBT.")

## Interpretation template for final write-up

After running the notebook, summarise your result like this:

- **Best model:** `[fill from comparison table]`
- **Test year:** `2025`
- **Key metric (MAE):** `[fill value]`
- **What drives demand most:** use the GBT feature-importance chart
- **When the model struggles:** use the residual-by-day/hour chart
- **Operational takeaway:** identify whether STN_0003 needs extra rebalancing attention during specific hours or days

## Recommended next upgrades

1. Add month / season cyclical features.
2. Add previous-day same-hour rolling statistics.
3. Add nearby-station demand spillover features.
4. Tune GBT hyperparameters with grid search.
5. Compare with Random Forest or XGBoost outside Spark if needed.